# Hands-On Session 2: Building out your first RAG
## Topics:
**Lab 2-A: Build and inspect**

1. Understanding the data, including the pages a parser gets wrong
2. Experiment with different chunking strategies
3. Creating our vector database

**Lab 2-B: Measure retrieval**

4. Measure retrieval against a golden question set

**Lab 2-C: Make your own test questions**

5. Generate questions from the hard pages, review them, and hold some out

**Dataset: CalHR employee benefits.** 20 California state employee benefits publications: 6 handbooks and 14 flyers, 186 pages.

<!-- twin: dataset-specific -->

**Goal.** After this lab you can build a naive RAG pipeline end to end, find the pages its parser gets wrong, measure whether retrieval hands the model the passage each question needs, and build and review a small test set of your own.

In [ ]:
#@title #Installs, Imports and Global Variables
%%capture
%pip install ipython-autotime
%load_ext autotime
# Every version here is pinned on purpose. An unpinned dependency is a bet that nothing
# upstream changes between now and the moment you run this, and that bet is how a notebook
# that worked last week stops working today.
%pip install -q \
  litellm==1.102.0 "openai>=2.20.0,<3.0.0" \
  llama-index==0.14.24 llama-index-readers-file==0.7.0 fonttools==4.65.0 \
  llama-index-vector-stores-chroma==0.6.0 \
  llama-index-embeddings-litellm==0.6.0 llama-index-llms-litellm==0.8.0 \
  chromadb==1.5.9 pypdf==6.19.0 pymupdf==1.28.2 \
  llama-index-retrievers-bm25==0.8.0 bm25s==0.3.14 PyStemmer==2.2.0.3 \
  ipywidgets==7.7.1 pandas==2.2.3 tiktoken==0.14.0

In [ ]:
#@title Imports
import os
import random
import pymupdf  # PyMuPDF
import io
import pickle
import json
import zipfile
import concurrent.futures

import numpy as np
import pandas as pd

from PIL import Image
from litellm import completion
from llama_index.core import SimpleDirectoryReader
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter

import litellm

litellm.drop_params = True

# Dollar amounts in the answers and contexts would otherwise be typeset as maths in table output.
pd.set_option("display.html.use_mathjax", False)
# Keep the model libraries quiet. At the default level they log every call, and a wall of INFO lines
# buries the one result you are looking for.
import logging
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

In [ ]:
# twin: dataset-specific
# Your OpenRouter key. One key covers every model call in this notebook: chat
# and embeddings both go through OpenRouter.
#
# Get a key at https://openrouter.ai/keys. Then either add it to Colab's Secrets
# panel (the key icon in the left sidebar) under the name OPENROUTER_API_KEY and
# switch on notebook access, or paste it when this cell asks. On the workshop day
# the hub handed one out instead, and it still will if a workshop is running.
import json, os, urllib.error, urllib.request
from getpass import getpass

WORKSHOP_HUB = "https://tdwi-workshop-hub.vercel.app"

def load_openrouter_key(hub=WORKSHOP_HUB):
    """Find a key and load it into this session. Says where it came from, never what it is."""
    if os.environ.get("OPENROUTER_API_KEY"):
        return "this session"
    try:
        from google.colab import userdata
        key = userdata.get("OPENROUTER_API_KEY")
        if key:
            os.environ["OPENROUTER_API_KEY"] = key.strip()
            return "Colab Secrets"
    except Exception:
        pass  # not in Colab, no secret by that name, or notebook access is switched off
    try:
        request = urllib.request.Request(
            f"{hub}/api/env", data=b"{}", headers={"content-type": "application/json"})
        with urllib.request.urlopen(request, timeout=20) as response:
            os.environ.update(json.load(response))
        return "the workshop hub"
    except Exception:
        pass  # the hub only hands out keys while a workshop is running
    key = getpass("Paste your OpenRouter API key (get one at https://openrouter.ai/keys): ").strip()
    if not key:
        raise SystemExit(
            "No key was pasted, so nothing in this notebook can call a model. Get one at "
            "https://openrouter.ai/keys, then run this cell again.")
    os.environ["OPENROUTER_API_KEY"] = key
    return "the key you pasted"

print("OpenRouter key loaded from", load_openrouter_key())

# Shared settings, fetched rather than retyped. Every lab reads the same file, so the chunk
# size here and the chunk size in Lab 3 cannot drift apart.
import json as _json, urllib.request as _urllib

DATASET = 'calhr'   # one notebook per dataset, on purpose; no picker here
_CONFIG_URL = "https://raw.githubusercontent.com/ndecavel/tdwi-workshop-labs/36c0a68ca026b555be4ad5175278173fc6ce407b/data/dataset_config.json"
_urllib.urlretrieve(_CONFIG_URL, "/content/dataset_config.json")
CFG = _json.load(open("/content/dataset_config.json"))
SHARED, DS = CFG['shared'], CFG['datasets'][DATASET]


MODEL = SHARED['litellm_chat_model']      # LiteLLM form, needs the openrouter/ prefix
EMBED_MODEL = SHARED['embed_model']

# Query-time embeddings must use the same model that built the index. A different one
# does not raise, it just quietly retrieves worse.

# Chunking comes from the dataset, not from this cell, so that what you build here matches
# the corpus Lab 3 downloads and the chunk sizes the workshop hub's explorer shows. Behr is
# 1024/200; CalHR is 384/96 because it is a sixth the size and needs finer chunks before
# retrieval has anything to discriminate on.
CHUNK_SIZE = DS['chunk_size']
CHUNK_OVERLAP = DS['chunk_overlap']
print(f"chunking at {CHUNK_SIZE}/{CHUNK_OVERLAP} for {DS['label']}")

DOCS_FOLDER_PATH = "/content/downloaded_pdfs/"

### Source documents for this dataset.
### Download the 20 CalHR benefits PDFs from the workshop repo. They are published as
### separate files rather than a zip, so this is a loop rather than an unzip.
CALHR_DOCS = [
    "2026-cash-options-flyer.pdf",
    "2026-coben-handbook.pdf",
    "2026-dental-active-handbook.pdf",
    "2026-dental-retiree-handbook.pdf",
    "2026-dependent-care-reimbursement-account-flyer.pdf",
    "2026-eap-core-flyer.pdf",
    "2026-eap-digital-emotional-wellbeing.pdf",
    "2026-eap-financial-wellbeing.pdf",
    "2026-eap-legal-services.pdf",
    "2026-eap-lifemart-discount-center.pdf",
    "2026-eap-manager-support.pdf",
    "2026-eap-website-flyer.pdf",
    "2026-eap-wellbeing-coaching.pdf",
    "2026-eap-work-life-services.pdf",
    "2026-flexelect-handbook.pdf",
    "2026-long-term-disability-benefits-guide.pdf",
    "2026-long-term-disability-flyer.pdf",
    "2026-medical-reimbursement-account-flyer.pdf",
    "flexelect-reimbursement-account-flyer.pdf",
    "2027-benefit-summary-guide-active-state-employees.pdf",
]
_DOCS_BASE = "https://raw.githubusercontent.com/ndecavel/tdwi-workshop-labs/36c0a68ca026b555be4ad5175278173fc6ce407b/data/calhr/"

os.makedirs(DOCS_FOLDER_PATH, exist_ok=True)
for _name in CALHR_DOCS:
    _urllib.urlretrieve(_DOCS_BASE + _name, DOCS_FOLDER_PATH + _name)

_got = len([f for f in os.listdir(DOCS_FOLDER_PATH) if f.endswith(".pdf")])
print(f"{_got} PDFs in {DOCS_FOLDER_PATH}")
assert _got == len(CALHR_DOCS), "A document failed to download. Re-run this cell."

____
## LAB 2-A: Build and inspect
### 1. Understanding the data
For this hands-on session we are working with the benefits publications California's Department of Human Resources (CalHR) gives to state employees. Twenty documents, 186 pages: six handbooks covering dental, CoBen, FlexElect and long-term disability, and fourteen two-page flyers covering the Employee Assistance Program and the reimbursement accounts.

Our focus is to build a natural-language question-answering interface that cites its source, so nobody has to already know which handbook a rule lives in before they can look it up.

With any problem, it is critical for us to view the data being inputted in the model to better understand.

<!-- twin: dataset-specific -->

____
### Which pages does the parser get wrong? (about 10 minutes)

Before we build anything, look at what a machine will be handed. Parsing fails on a minority of pages, and they are often the pages people ask about most: rate tables, compliance grids, scanned sheets. Finding them takes no model calls. Scoring all 186 pages takes about ten seconds.

1. **Score** every page with PyMuPDF alone, using the rules in the table below. The rules are plain on purpose, so you can argue with them.
2. **List** the ten highest-scoring pages.
3. **Look** at each one beside its text, and say whether the text is complete, the tables are intact and the reading order is right. Looking at the page next to its text is the only check that catches a table whose numbers all survived but whose columns no longer line up.

Two readers appear below. PyMuPDF does the scoring. The index you build later in this lab is fed by a different reader, pypdf, so you will see both. They can disagree. A page can be hard for one extractor and fine for another (the same "fi" ligature comes out as one odd character in PyMuPDF, as a glyph name such as `/f_i` in pypdf, and as a missing letter in older PyMuPDF versions), which is why you check what **your** pipeline reads. The ingestion step below puts those glyph-name letters back before anything is indexed, so the viewer shows the text after that repair.

<!-- twin: dataset-specific -->

In [ ]:
#@title Score every page with PyMuPDF only
import re, collections
from tqdm.auto import tqdm

# The rules, in one table you can argue with. A page's score is the sum of the points
# for every signal that fires. Nothing here calls a model.
RULES = pd.DataFrame([
    ("no text layer (scan)",   "6",     "under 30 characters of text but more than 40% of the page is picture: the words exist only as pixels"),
    ("mostly picture",         "2",     "30 to 300 characters and more than 30% picture: a flyer whose text is mostly baked into images"),
    ("tables found",           "1 each, up to 3", "PyMuPDF's find_tables() sees a grid: rows and columns are about to be flattened into lines"),
    ("merged header cells",    "1",     "a table cell spans several columns: a plan or product name that applies to a whole block of numbers"),
    ("wrapped cell (4+ lines)", "3",    "a cell holds four or more lines: reading order inside the cell is easy to get wrong"),
    ("number-dense lines",     "2, or 3 if over 40%", "15% or more of the lines are mostly digits: rates and limits, where a shifted column changes the answer"),
    ("ligature glyphs / dropped letters", "2, +1 for 5 or more, +1 for 20 or more",
                               "a fi or fl ligature came out as one odd character, or lost its letter"),
    ("broken characters",      "2",     "replacement characters, private-use glyphs or '(cid:123)' where a font could not be decoded"),
    ("letters swapped inside words", "2", "a look-alike letter from another alphabet inside an ASCII word, such as a Greek capital epsilon in LEED"),
], columns=["signal", "points", "why it matters"])

LIGATURES = re.compile(r"[\ufb00-\ufb06]")                           # fi, fl, ff... kept as one character
BROKEN = re.compile(r"\ufffd|[\ue000-\uf8ff]|\(cid:\d+\)")          # undecodable font output
SWAPPED = re.compile(r"\b[A-Za-z]*[\u00c0-\u024f\u0370-\u03ff][A-Za-z]+\b|\b[A-Za-z]+[\u00c0-\u024f\u0370-\u03ff]\b")

# One short function per signal. Each takes the measurements `m` for one page and returns points.
def no_text_layer(m):    return 6 if m["chars"] < 30 and m["picture"] > 0.4 else 0          # a scan
def mostly_picture(m):   return 2 if m["picture"] > 0.3 and 30 <= m["chars"] < 300 else 0   # text baked into images
def tables(m):           return min(m["tables"], 3)                                         # grids to flatten
def merged_headers(m):   return int(m["merged_cells"] > 0)                                  # a header spanning columns
def wrapped_cells(m):    return 3 if m["tallest_cell"] >= 4 else 0                          # long text inside one cell
def number_dense_lines(m):                                                                  # rate and limit tables
    return (m["numeric_share"] >= 0.15 and m["lines"] >= 8) * (2 + (m["numeric_share"] >= 0.4))
def ligature_trouble(m):                                                                    # fi and fl glyphs, dropped letters
    n = m["ligatures"] + m["lost_letters"]
    return (n >= 1) * 2 + (n >= 5) + (n >= 20)
def broken_characters(m): return 2 if m["broken"] > 0 else 0                                # undecodable glyphs
def swapped_letters(m):   return 2 if m["swapped"] > 0 else 0                               # look-alike letters in words

SIGNALS = {"no text layer (scan)": no_text_layer, "mostly picture": mostly_picture,
           "tables found": tables, "merged header cells": merged_headers,
           "wrapped cell (4+ lines)": wrapped_cells, "number-dense lines": number_dense_lines,
           "ligature glyphs / dropped letters": ligature_trouble,
           "broken characters": broken_characters, "letters swapped inside words": swapped_letters}

def measure(page, text, vocab):
    """Everything the signals need to know about one page."""
    lines = [l for l in text.splitlines() if len(l.strip()) > 2]
    grids = page.find_tables().tables
    area = page.rect.get_area()
    # A word lost a ligature if putting an i or l back after one of its f's makes a word the
    # corpus uses elsewhere (a missing i after an f, say).
    words = re.findall(r"[a-z]{4,}", text.lower())
    lost = sum(any(vocab[w[:f.end()] + x + w[f.end():]] >= 2 for f in re.finditer("f", w) for x in "il")
               for w in words if "f" in w and vocab[w] < 200)
    return dict(
        chars=len(text.strip()), lines=len(lines),
        picture=sum((pymupdf.Rect(b["bbox"]) & page.rect).get_area() for b in page.get_image_info()) / area,
        tables=len(grids),
        merged_cells=sum(c is None for t in grids for r in t.rows for c in r.cells),
        tallest_cell=max([c.count("\n") + 1 for t in grids for r in t.extract() for c in r if c] or [0]),
        numeric_share=sum(sum(ch.isdigit() for ch in l) > 0.4 * len(l) for l in lines) / max(len(lines), 1),
        ligatures=len(LIGATURES.findall(text)), lost_letters=lost,
        broken=len(BROKEN.findall(text)), swapped=len(SWAPPED.findall(text)))

def score_all_pages(folder):
    paths = sorted(os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".pdf"))
    # Pass 1: read every page's text once, and count the words in the whole corpus.
    texts, vocab = {}, collections.Counter()
    for path in paths:
        with pymupdf.open(path) as pdf:
            for i, page in enumerate(pdf):
                texts[(os.path.basename(path), i + 1)] = page.get_text()
                vocab.update(re.findall(r"[a-z]{4,}", texts[(os.path.basename(path), i + 1)].lower()))
    # Pass 2: measure and score each page. find_tables() is the slow part.
    rows = []
    with tqdm(total=len(texts), desc="Scoring pages") as bar:
        for path in paths:
            with pymupdf.open(path) as pdf:
                for i, page in enumerate(pdf):
                    name = os.path.basename(path)
                    m = measure(page, texts[(name, i + 1)], vocab)
                    points = {label: fn(m) for label, fn in SIGNALS.items()}
                    rows.append(dict(doc=name, page=i + 1, score=sum(points.values()),
                                     reasons="; ".join(k for k, v in points.items() if v), tables=m["tables"]))
                    bar.update(1)
    return pd.DataFrame(rows).sort_values("score", ascending=False, kind="stable").reset_index(drop=True)

def left_aligned(table):
    """Show a table with its text lined up on the left, where it is easier to read."""
    return table.style.set_properties(**{"text-align": "left"}).set_table_styles(
        [{"selector": "th", "props": [("text-align", "left")]}])

page_scores = score_all_pages(DOCS_FOLDER_PATH)
print(f"Scored {len(page_scores)} pages from {page_scores['doc'].nunique()} documents.")
print("Pages by score:", dict(sorted(page_scores["score"].value_counts().items(), reverse=True)))
pd.set_option("display.max_colwidth", None)
left_aligned(RULES)

In [ ]:
#@title The ten highest-scoring pages
def top_pages(scores, n=10, per_doc=4, per_problem=3):
    """The n best-scoring pages. Without caps, near-identical sheets fill the list, so keep at most
    per_doc pages from one document and per_problem pages with the same reasons and table count."""
    keep, from_doc, from_problem = [], collections.Counter(), collections.Counter()
    for i, row in scores.iterrows():
        problem = (row["reasons"], row["tables"])
        if from_doc[row["doc"]] >= per_doc or from_problem[problem] >= per_problem:
            continue
        from_doc[row["doc"]] += 1
        from_problem[problem] += 1
        keep.append(i)
    return scores.loc[keep[:n]].reset_index(drop=True)

flagged = top_pages(page_scores)
left_aligned(flagged[["doc", "page", "score", "reasons"]])

### Look at the pages

Use **Previous** and **Next** to move through the ten pages. Page numbers count pages in the PDF file, which can differ from the number printed on the page. Each shows the page itself, the text this lab will actually index for it, and the text PyMuPDF read. For each page, tick what is true (text complete, tables intact, reading order right), add a short note, and press **Save verdict**. Nothing is ticked to begin with, so a page you have not judged is not a page you have passed. **Show my verdicts** prints your table, `page_verdicts`, which Lab 2-C reads.

In [ ]:
#@title Look at each flagged page beside its text, and record a verdict
import base64, html
from ipywidgets import Button, Checkbox, HBox, HTML, Layout, Output, Text, VBox
from IPython.display import display

# Two different readers touch every page, and they can disagree. The scorer used PyMuPDF.
# The index you build below is fed by pypdf, through SimpleDirectoryReader. Show both.
_indexed_text, _page_images = {}, {}

def clean_pypdf_text(text):
    """pypdf writes a ligature it cannot map as the glyph's name, so "benefit" arrives as
    "bene/f_it". Put the letters back, or the word can never match a question that spells it."""
    return re.sub(r"/((?:f_)+[a-z])", lambda m: m.group(1).replace("_", ""), text)

def indexed_page_text(doc, page_no):
    """The text this lab will index for a page: SimpleDirectoryReader's PDF reader, one Document per page, repaired."""
    if doc not in _indexed_text:
        pages = SimpleDirectoryReader(input_files=[os.path.join(DOCS_FOLDER_PATH, doc)]).load_data()
        _indexed_text[doc] = {n + 1: clean_pypdf_text(p.text) for n, p in enumerate(pages)}
    return _indexed_text[doc].get(page_no, "")

def pymupdf_page_text(doc, page_no):
    """The text PyMuPDF read from the page, which is all the scorer above ever saw."""
    with pymupdf.open(os.path.join(DOCS_FOLDER_PATH, doc)) as pdf:
        return pdf[page_no - 1].get_text()

def page_image_b64(doc, page_no):
    """The page as a picture, base64 encoded so it can sit inside an HTML widget."""
    if (doc, page_no) not in _page_images:
        with pymupdf.open(os.path.join(DOCS_FOLDER_PATH, doc)) as pdf:
            png = pdf[page_no - 1].get_pixmap(dpi=110).tobytes("png")
        _page_images[(doc, page_no)] = base64.b64encode(png).decode()
    return _page_images[(doc, page_no)]

PANE = ("white-space:pre-wrap;max-height:260px;overflow:auto;margin:4px 0 12px;padding:8px;"
        "border:1px solid #888;background:rgba(127,127,127,0.12);font:12px/1.4 monospace")

def text_pane(text):
    return f'<pre style="{PANE}">{html.escape(text) if text.strip() else "(no text at all)"}</pre>'

def page_html(doc, page_no):
    return f"""
<div style="display:grid;grid-template-columns:minmax(0,1fr) minmax(0,1fr);gap:16px;align-items:start;font:13px sans-serif">
  <div><b>The page</b><br>
    <img src="data:image/png;base64,{page_image_b64(doc, page_no)}" style="width:100%;border:1px solid #888"></div>
  <div>
    <b>Text this lab will index</b> (pypdf, through SimpleDirectoryReader)
    {text_pane(indexed_page_text(doc, page_no))}
    <b>PyMuPDF's text</b> (what the scorer read)
    {text_pane(pymupdf_page_text(doc, page_no))}
  </div>
</div>"""

# Your verdicts, one row per page, kept for the rest of the session. Lab 2-C reads this table.
VERDICT_COLUMNS = ["doc", "page", "score", "reasons", "text_complete", "tables_intact", "reading_order_right", "note"]
_verdict_rows = {}
page_verdicts = pd.DataFrame(columns=VERDICT_COLUMNS)

viewer, heading = HTML(), HTML()
text_ok = Checkbox(description="Text is complete", indent=False)
tables_ok = Checkbox(description="Tables are intact", indent=False)
order_ok = Checkbox(description="Reading order is right", indent=False)
note = Text(placeholder="A short note: what went wrong, or what to look at", layout=Layout(width="95%"))
status = HTML()
verdict_output = Output()
shown = {"i": 0}

def show_page(i):
    shown["i"] = i
    row = flagged.iloc[i]
    heading.value = (f"<b>Page {i + 1} of {len(flagged)}: {html.escape(row['doc'])}, page {row['page']}</b>"
                     f" (score {row['score']}: {html.escape(row['reasons'])})")
    viewer.value = page_html(row["doc"], int(row["page"]))
    saved = _verdict_rows.get((row["doc"], int(row["page"])))
    text_ok.value, tables_ok.value, order_ok.value = (saved["text_complete"], saved["tables_intact"],
                                                      saved["reading_order_right"]) if saved else (False,) * 3
    note.value = saved["note"] if saved else ""
    status.value = "Verdict saved for this page." if saved else "Tick what is true, then save."

def save_verdict(_=None):
    global page_verdicts
    row = flagged.iloc[shown["i"]]
    _verdict_rows[(row["doc"], int(row["page"]))] = dict(
        doc=row["doc"], page=int(row["page"]), score=int(row["score"]), reasons=row["reasons"],
        text_complete=text_ok.value, tables_intact=tables_ok.value,
        reading_order_right=order_ok.value, note=note.value)
    page_verdicts = pd.DataFrame(list(_verdict_rows.values()), columns=VERDICT_COLUMNS)
    status.value = f"Saved. {len(page_verdicts)} page(s) have a verdict."

def show_verdicts(_=None):
    with verdict_output:
        verdict_output.clear_output(wait=True)
        display(page_verdicts if len(page_verdicts) else "No verdicts saved yet.")

def step(by):
    show_page((shown["i"] + by) % len(flagged))

previous_button, next_button = Button(description="Previous"), Button(description="Next")
save_button, show_button = Button(description="Save verdict", button_style="success"), Button(description="Show my verdicts")
previous_button.on_click(lambda _: step(-1))
next_button.on_click(lambda _: step(1))
save_button.on_click(save_verdict)
show_button.on_click(show_verdicts)

show_page(0)
display(VBox([HBox([previous_button, next_button]), heading, viewer,
              HBox([text_ok, tables_ok, order_ok]), note, HBox([save_button, show_button, status]),
              verdict_output]))

### Where these pages go next

These pages are where test questions should come from. Questions written from random pages under-sample exactly the tables and scans people ask about, so Lab 2-C offers these pages as the ones to generate from. They are also where Lab 3's parser should be judged: a better parser has to win on these pages, not on the easy ones.

The workshop hub's Parsing page (https://tdwi-workshop-hub.vercel.app/parsing) shows the same kind of page read by PyMuPDF and by LlamaParse, side by side.

The pages above come from the 20 CalHR benefits publications. Six of them are handbooks, 13 to 39 pages each, and they carry 90% of the text in the corpus. The other 14 are two-page flyers.

Visually this layout is clear to a human reader. It has headed sections and tables of rates, limits and deadlines, which let you find a number quickly. That clarity is mostly structural, and structure is the first thing chunking destroys. A premium table means nothing once it is cut away from the plan name printed above it. We will go over specific solutions to this in our next lab. For now, we can focus on simpler chunking and embedding methodologies to set up our baseline.

____
## Brief aside: Business Relevance

When integrating data into your RAG solution, ensure it's tied to your business problem. For a benefits desk, these documents cover the questions that arrive every open enrollment season:
- **Eligibility:** "How long after my hire date do I have to enroll in dental?"
- **Cost:** "What do I pay for a non-PPO dentist under the indemnity plan?"
- **Process:** "How do I submit a reimbursement claim, and by when?"

Answering those from the source document, with a citation, is the difference between a help desk and a rumour.

**I challenge anyone building out a RAG solution to have a clear idea of the questions that solution is hoping to answer. This exercise will allow you to not only filter the necessary data to use in the solution, but to help justify where the solution is delivering value to the business.**

Two traps are already baked into this corpus, on purpose. Nineteen documents are for 2026 and one, the Benefit Summary Guide, is for 2027. And the two dental handbooks, active and retiree, share about half their text word for word. Right topic and wrong document is the most expensive kind of wrong, because the answer still looks correct.



Okay, now back to building out first RAG pipeline!
_____

To start, we will experiment with a few simple chunking strategies to help create our first vector store.

## Instructions:
1. Open the **Chunking** page on the workshop hub: https://tdwi-workshop-hub.vercel.app/chunking
2. Try both splitters, move the chunk size and overlap, and compare the results against the same document.

🤔 Do you notice any issues with the documents and the chunks being displayed? If you were assigned a task that relied on this data, but were provided only the chunk of text information and not the pdf itself, would you be able to make sense of it?
---

<!-- twin: dataset-specific -->

* * *
### The chunking explorer lives on the workshop hub

Open https://tdwi-workshop-hub.vercel.app/chunking in a second tab. It runs the same two
splitters, `SentenceSplitter` and `TokenTextSplitter`, over the same documents you are about
to index, and it offers both corpora. Nothing to install and nothing to launch, so a broken
cell here cannot cost you the exercise.

Leave it open. You will want it again in Lab 3, when the chunk size stops being a setting
you picked and starts being a number you can defend.


____

As we can see, the chunks cut straight across the structure these documents rely on. A table of rates gets separated from the plan name that gives it meaning, and a list of eligibility rules loses the heading that said who it applies to. To start, we will use the SentenceSplitter as our more naive approach in order to measure the improvements of more advanced techniques in the next hands-on session.


The following code will go through each of our PDFs and chunk their information into a vector database. In this example, we will use simply use llama index with an ephemeral chromadb. Note: There are many ways to store the data, and it is important to understand the pros/cons of each when deciding which to use in production. In particular, if your system uses postgres, you may be better suited sticking to using pgvector versus having to maintain a new database altogether. We will also use llama_index ingestion pipelines to run a job that embeds each chunk into our dataframe.
In addition, we will use the text-embedding-3-small embedding model (https://platform.openai.com/docs/guides/embeddings).

**Two repairs before the chunks are cut.** The ingestion cell below repairs what pypdf hands over in two ways, and 2-B measures what they were worth. First, by default llama-index writes `page_label: 1` and `file_path: /content/downloaded_pdfs/...` into the text of every chunk it embeds, so every vector carries the same lines and the chunks look a little more alike than they are. The file name and page stay on each chunk, so you can still cite sources, but they are kept out of the text that is embedded and searched. On this corpus that took dense retrieval from 56.1% to 61.4% hit@5 and from 0.403 to 0.450 MRR (1 run, 2026-10-21). With 57 questions that gain is inside the noise: the 95% interval for the hit@5 change runs from a loss of 3.5 points to a gain of 14.0. Second, when pypdf cannot map a ligature it writes the glyph's name, so "benefit" arrives as `bene/f_it`. Here that happens 114 times, 107 of them in the long term disability guide, and the cell puts the letters back. That alone moved dense MRR by 0.009 and nothing else (1 run, 2026-10-21): small, but a word that can never match a question is a defect whatever it costs.

<!-- twin: dataset-specific -->

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from IPython.display import Markdown, display
from llama_index.embeddings.litellm import LiteLLMEmbedding

import chromadb

from llama_index.core import Settings

# Set the embedding model globally, before any index is built.
# Without this, VectorStoreIndex falls back to Settings.embed_model, which defaults to
# OpenAI and needs an OPENAI_API_KEY we deliberately no longer have. Worse, when that key
# IS present, retrieval quietly runs on a different model than the one that built the
# index, and nothing raises. The same model on both sides, or the numbers are fiction.
Settings.embed_model = LiteLLMEmbedding(model_name=EMBED_MODEL, embed_batch_size=100)


In [ ]:
#@title Data ingestion pipeline (about 1-3 minutes)
# twin: dataset-specific

## Step 0. Initialize chromadb
chroma_client = chromadb.EphemeralClient()
chroma_collection = chroma_client.get_or_create_collection(
                    name="initial_collection",
                    configuration={"hnsw": {"space": "cosine"}}
                )
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

## Step 1. Use llama-index to create a DirectoryReader
# 20 PDFs, one document per page. If this prints far fewer documents than pages,
# llama-index-readers-file is missing and the text you get back will be junk.
documents = SimpleDirectoryReader(DOCS_FOLDER_PATH).load_data(num_workers=-2)
print(f"Loaded {len(documents)} pages from {DOCS_FOLDER_PATH}")

# Two repairs before chunking, both measured against the golden set in 2-B.
for d in documents:
    d.set_content(clean_pypdf_text(d.text))
    # Keep the file name and page on every chunk, for citing sources, but out of the text that is
    # embedded and searched. By default llama-index writes "page_label: 1" and "file_path: /content/..."
    # into every chunk it embeds, so every vector carries the same lines.
    d.excluded_embed_metadata_keys = list(d.metadata)
    d.excluded_llm_metadata_keys = list(d.metadata)
# A page with no text layer (a scan) would now embed an empty string, which the embedding API refuses.
documents = [d for d in documents if d.text.strip()]
print(f"{len(documents)} pages have text to index")

## Step 2. Chunk and embed.
# embed_batch_size matters more than it looks: the default of 10 sends ~200 separate
# requests for this corpus. Batching at 100 cuts the wall time by about 4x.
pipeline = IngestionPipeline(
    transformations=[SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP),
                     LiteLLMEmbedding(model_name=EMBED_MODEL, embed_batch_size=100)
                     ])
nodes = pipeline.run(documents=documents, num_workers=-2, show_progress=True)

## Step 3. Write to Chroma in batches.
# Chroma refuses a single .add() larger than 5461 records. This corpus is under that,
# but a bigger one would fail at the very end of a long, expensive cell.
BATCH = 4000
for i in range(0, len(nodes), BATCH):
    vector_store.add(nodes[i:i + BATCH])

## Step 4. Say out loud what landed, so an empty index cannot masquerade as a working one.
count = chroma_collection.count()
print(f"Indexed {count} chunks from {len(documents)} pages")
assert count > 0, "Nothing was indexed. Re-run this cell before continuing; every answer below would be invented."

# Keep the indexed chunks under their own name. Later cells retrieve into other variables, and
# Lab 2-C builds its keyword search over exactly these chunks.
index_nodes = nodes

In [ ]:
# embed_model is passed explicitly as well as set globally above. Queries have to be
# embedded by the same model that embedded the corpus.
index = VectorStoreIndex.from_vector_store(vector_store, embed_model=Settings.embed_model)


____
Now that we have created/loaded in our vector store, we can create our query engine. To start, we will simply extract the top 5 documents

In [ ]:
# twin: dataset-specific
from llama_index.core.prompts import RichPromptTemplate

retriever = index.as_retriever(similarity_top_k=SHARED['similarity_top_k'])

template_str = """We have provided context information below.
---------------------
{{ context_str }}
---------------------
Given this information, please answer the question: {{ query_str }}
"""

qa_template = RichPromptTemplate(template_str)

query_str = "How many days after my hire date do I have to enroll in health and dental benefits?" #@param {"type": "string"}

retrieved = retriever.retrieve(query_str)
context_str = "\n\n".join([n.node.get_content() for n in retrieved])
messages_chat = [
            {"content": qa_template.format(context_str=context_str, query_str=query_str), "role": "user"}
        ]
response = completion(model=MODEL, messages=messages_chat, temperature=1, stream=False)
response_text = response.choices[0].message.content
display(Markdown(f"**RESPONSE:**\n\n{response_text}"))

### Sources used in the response:
for i, node in enumerate(retrieved):
  display(Markdown(f"**Source {i+1}: {node.node.metadata['file_name']} -> Similarity Score = {node.score:.2f}**"))
  display(Markdown(node.node.text))

____
## Congrats on creating your first RAG QA pipeline 🎉

Feel free to experiment with different questions above! In particular, take a look at a pdf and try to see how well it retrieves the correct information.

**Questions to try.** Paste any of these into `query_str` above and re-run the cell.

* How many EAP counseling sessions am I entitled to?
* When do Long Term Disability benefits terminate?
* What is the waiting period before Long Term Disability payments begin?
* What does a Dependent Care Reimbursement Account cover?
* What do I pay out of pocket when I see a dentist who is not in the network?

Then try the two traps. Ask a question about retiree dental coverage and check whether the retiree handbook or the active one came back. Ask about a 2027 rate and see which year the source document is from. Watch the sources printed under each answer as much as the answer itself.

<!-- twin: dataset-specific -->

stop.svg

**STOP** - end of LAB 2-A


---



## LAB 2-B: Measure retrieval

### Importance of Metric-Driven Experimentation in Developing a RAG System
There are many ways to measure the success of a RAG system, and metric-driven experimentation is a crucial part of this process. While user testing is essential, it doesn't scale well for rapid experimentation. A golden dataset with questions, context, and answers lets you compare strategies objectively and make meaningful improvements efficiently. Though a perfect golden dataset is rarely available, generating your own version with context-specific questions and answers can provide valuable insights.

This lab scores retrieval only. Each row of the golden set carries a question, the passage the question was written from (`context`) and the file that passage sits in (`source_file`), so for each question we can ask one thing: did the passage that answers it reach the model?

**Why there is no judge model here.** A judge reads answers, costs a model call per question per metric, and its scores move between runs. Checking whether a passage arrived needs only text matching. So every question is scored, not a sample, the only model call is one batched embedding of the questions, and the same question gets the same score every time.

If you are interested in understanding how I created this sample dataset, please refer to the synthetic dataset documentation for RAG here: https://deepeval.com/docs/golden-synthesizer


In [ ]:
# The golden set URL comes from the dataset config fetched at the top of this notebook.
_ = _urllib.urlretrieve(DS['golden_url'], "/content/Golden_Test_Data_DeepEval.csv")


In [ ]:
golden_df = pd.read_csv('/content/Golden_Test_Data_DeepEval.csv')
golden_df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'], inplace=True)
print(golden_df.shape)
print("Example inputs and expected_outputs from the golden dataset:",end='\n\n')
for example in golden_df.sample(3)[['input', 'expected_output']].values:
  print(f"Input: {example[0]}")
  print(f"Expected Output: {example[1]}", end='\n\n')

In [ ]:
import ast

# "evolutions" is the style the generator was told to write each question in. It is stored as
# text that looks like a list, so turn it back into a list before counting.
styles = golden_df["evolutions"].apply(ast.literal_eval)
style_counts = styles.explode().value_counts()
print("Question styles in the golden set:")
print(style_counts.to_string())

# "synthetic_input_quality" is the generator's own score for each question.
print("\nThe generator's own quality score for each question:")
print(golden_df["synthetic_input_quality"].value_counts().sort_index(ascending=False).to_string())
kept = (golden_df["synthetic_input_quality"] == SHARED["quality_filter"]).sum()
print(f"\n{kept} of {len(golden_df)} questions score {SHARED['quality_filter']}. Lab 2 and Lab 3 both score every one of those.")

# One question each from three different styles.
print("\nThree questions, three styles:")
for style in ["Hypothetical", "Comparative", "Concretizing"]:
    question = golden_df[styles.apply(lambda s: s == [style])]["input"].iloc[0]
    print(f"\n[{style}] {question}")

**Would a real user type this?** Read the three questions above. Generated styles such as "Imagine a scenario where..." rarely match how people ask, and every metric in this lab is only as honest as the questions it is scored on. Keep that in mind: Lab 2-C has you generate, read and review a few questions of your own.

### The four numbers

Every question the generator scored 1.0 is scored here, not a sample. For each one the retriever returns its top k chunks (k is 5), and we ask whether those chunks hold the passage the question was written from:

* **hit@k**: some chunk holds the passage.
* **source hit@k**: some chunk from the question's own file holds the passage. Right file, wrong place does not count, and neither does the right words in the wrong file.
* **recall@k**: the share of the passage the k chunks carry between them. RAGAS called this context recall; here it is measured without a judge.
* **MRR**: 1 divided by the rank of the first chunk that holds the passage, or 0 if none of the k does, averaged over the questions.

**What "holds" means.** Strip the markup, lowercase everything, and cut the passage into overlapping 6-word phrases. A chunk holds the passage when it contains at least half of those phrases. The phrase length and the one-half threshold both come from `SHARED['retrieval_metrics']`, the same settings Lab 3 reads.

The next cell defines the scoring. It is shared with Lab 3 word for word, so a number here means exactly what it means there. The only model call in it is one batched embedding of the questions.


In [ ]:
## Retrieval metrics. Lab 2 and Lab 3 run this same cell, word for word, so a score in one lab
## means exactly what it means in the other. A check in the workshop repo fails if the copies differ.
## Nothing here calls a model except embed_questions, which makes one batched call per 100 questions.
import ast, functools, os, re, unicodedata
import numpy as np
import pandas as pd
import tiktoken
from llama_index.core import Settings
from llama_index.core.schema import QueryBundle

RM = SHARED['retrieval_metrics']     # shingle_size, hit_threshold, depths, ci_resamples
ENCODING = tiktoken.get_encoding("o200k_base")   # the tokenizer of the gpt-4.1 family the labs generate with

@functools.lru_cache(maxsize=None)
def words_of(text):
    """Lowercase words, with markup and punctuation out of the way, so a PDF line and a markdown table compare."""
    text = re.sub(r"</?[A-Za-z][A-Za-z0-9]*(?:\s[^<>]*)?/?>", " ", text)   # HTML tags only; a bare "< 50" is text
    text = re.sub(r"!?\[([^\]]*)\]\([^)]*\)", r"\1", text)               # links and images keep their label
    text = re.sub(r"^\s*\|?[\s:|-]{3,}\|?\s*$", " ", text, flags=re.M)    # markdown table rule rows
    text = re.sub(r"[|#*_`>~]", " ", text)                                  # table pipes, headers, emphasis
    text = unicodedata.normalize("NFKC", text).lower()                      # a ligature glyph becomes its letters
    text = re.sub(r"[-\u2010-\u2015]", " ", text)                           # hyphens and dashes split words
    return tuple(re.findall(r"[^\W_]+", text))

@functools.lru_cache(maxsize=None)
def phrases_of(text):
    """Every run of shingle_size consecutive words in the text."""
    words, n = words_of(text), RM['shingle_size']
    return frozenset(words[i:i + n] for i in range(len(words) - n + 1))

def share_found(passage, texts):
    """The share of the passage's phrases that appear in at least one of the texts. A passage too
    short to have a phrase counts as found (1.0) when its words appear, in order, in one text."""
    wanted = phrases_of(passage)
    if not wanted:
        words = " ".join(words_of(passage))
        return float(bool(words) and any(f" {words} " in f" {' '.join(words_of(t))} " for t in texts))
    return len(wanted & frozenset().union(*map(phrases_of, texts))) / len(wanted)

@functools.lru_cache(maxsize=None)
def tokens_in(text):
    return len(ENCODING.encode(text))

def holds(text, passage):
    """A chunk holds the passage when at least hit_threshold of the passage's phrases are in it."""
    return share_found(passage, [text]) >= RM['hit_threshold']

def passage_of(context_cell):
    """The golden `context` column is a stringified list of passages; join them into one."""
    return " ".join(str(p) for p in ast.literal_eval(context_cell))

def golden_questions(golden_df):
    """Every golden question the generator scored quality_filter, with its passage and source file."""
    frame = golden_df.rename(columns={'input': 'question', 'expected_output': 'ground_truth'})
    frame = frame[frame['synthetic_input_quality'] == SHARED['quality_filter']].reset_index(drop=True)
    return frame.assign(passage=frame['context'].map(passage_of),
                        source=frame['source_file'].map(os.path.basename))

def embed_questions(questions):
    """The questions as query bundles with their vectors filled in. The embedding model reads a
    question and a passage the same way, so one batch call gives the vectors one call each would."""
    questions = list(questions)
    vectors = Settings.embed_model.get_text_embedding_batch(questions)
    return [QueryBundle(query_str=q, embedding=v) for q, v in zip(questions, vectors)]

def score_retrieval(ranked, passage, source, k):
    """Score the top k of one ranked list of (file name, chunk text) pairs against one question."""
    top = ranked[:k]
    first = next((i for i, (_, text) in enumerate(top, 1) if holds(text, passage)), None)
    return {"hit": first is not None,                                         # a chunk holds the passage
            "source hit": any(f == source and holds(t, passage) for f, t in top),   # ... from the question's own file
            "recall": share_found(passage, [text for _, text in top]),        # share of the passage the model is handed
            "MRR": 1 / first if first else 0.0,                               # 1 / rank of the first chunk that holds it
            "tokens": sum(tokens_in(text) for _, text in top)}                # what the k chunks cost to send

def score_engines(engines_at, frame, bundles, text_of, depths=None):
    """One row per engine, depth and question. engines_at(k) returns {name: retriever}, and
    text_of(result) the chunk text to score, which is always the text the document said."""
    rows = []
    for k in depths or RM['depths']:
        for engine, retriever in engines_at(k).items():
            for n, (bundle, passage, source) in enumerate(zip(bundles, frame['passage'], frame['source']), 1):
                ranked = [(r.node.metadata['file_name'], text_of(r)) for r in retriever.retrieve(bundle)]
                rows.append({"engine": engine, "k": k, "Q #": n, **score_retrieval(ranked, passage, source, k)})
    return pd.DataFrame(rows)

def summary_at(scores, k):
    """Mean of each metric at depth k, one row per engine, in the order the engines were scored.
    hit, source hit and recall are percentages; MRR is out of 1.0."""
    table = scores[scores['k'] == k].groupby('engine', sort=False)[["hit", "source hit", "recall", "MRR"]].mean()
    table[["hit", "source hit", "recall"]] *= 100
    return table.round({"hit": 1, "source hit": 1, "recall": 1, "MRR": 3}).add_suffix(f"@{k}")

def by_depth(scores, engine):
    """One engine's metrics at every depth k: percentages, MRR out of 1.0, mean tokens of chunk text."""
    table = scores[scores['engine'] == engine].groupby('k')[["hit", "source hit", "recall", "MRR", "tokens"]].mean()
    table[["hit", "source hit", "recall"]] *= 100
    return table.round({"hit": 1, "source hit": 1, "recall": 1, "MRR": 3, "tokens": 0}).astype({"tokens": int})

def paired_interval(a, b):
    """The mean of a - b over the same questions, and a 95% interval for it: resample the
    questions with replacement ci_resamples times and keep the middle 95% of the means."""
    diff = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    draws = np.random.default_rng(SHARED['random_state']).integers(0, len(diff), (RM['ci_resamples'], len(diff)))
    means = diff[draws].mean(axis=1)
    return diff.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

def files_holding(frame, chunks):
    """For each question, the files whose chunks hold its passage. chunks: (file name, text) pairs."""
    return [sorted({f for f, t in chunks if holds(t, p)}) for p in frame['passage']]

In [ ]:
#@title Score the index on every golden question
import json

# Every question the generator scored 1.0, with the passage it was written from and its file.
frame = golden_questions(golden_df)
bundles = embed_questions(frame['question'])    # the one batched model call in this section
TOP_K = SHARED['similarity_top_k']

def engines_at(k):
    """The retrievers to score, asked for the top k chunks. Lab 2-C adds a second one."""
    return {"naive (dense)": index.as_retriever(similarity_top_k=k)}

# text_of picks the chunk text to score: what the document said, without the file name and page.
scores = score_engines(engines_at, frame, bundles, text_of=lambda r: r.node.get_content())
print(f"Scored {len(frame)} golden questions at k = {RM['depths']}.")

summary = summary_at(scores, TOP_K)
display(summary)

print("\nThe same index at deeper cuts: more chunks carry more of the passage, and cost more tokens to send.")
display(by_depth(scores, "naive (dense)"))

# Where the passages are. A passage that no chunk holds can never be retrieved, whatever the retriever.
holders = files_holding(frame, [(n.metadata['file_name'], n.get_content()) for n in index_nodes])
held = sum(1 for files in holders if files)
spread = sum(1 for files in holders if len(files) > 1)
print(f"\n{held} of {len(frame)} passages are held by some chunk in this index. That is the ceiling for hit@k.")
print(f"{spread} of them are held by more than one file, which is why hit@k runs ahead of source hit@k.")

print("RETRIEVAL " + json.dumps({"n": len(frame), "k": TOP_K,
                                 **{name: round(float(value), 3) for name, value in summary.iloc[0].items()},
                                 "held": held, "in_several_files": spread}))


In [ ]:
#@title The same 15 questions, one row each
# Exactly the questions Lab 3 shows: same filter, same size, same seed. The first 60 characters
# of each are printed so you can compare them with Lab 3's list.
sample = frame.sample(SHARED['lab_sample_size'], random_state=SHARED['random_state'])
for n, q in enumerate(sample['question'], start=1):
    print(f"{n:>2}. {q[:60]}")

# A mean hides the misses. Read the rows where the passage did not arrive, and ask whether the
# question or the retriever is at fault. score_engines numbers the questions 1..N in frame order.
at_k = scores[(scores['k'] == TOP_K) & (scores['engine'] == "naive (dense)")].set_index('Q #')
rows = []
for n, i in enumerate(sample.index, start=1):
    result = at_k.loc[i + 1]
    rows.append({"Q #": n,
                 "question": sample.loc[i, 'question'][:70] + ("..." if len(sample.loc[i, 'question']) > 70 else ""),
                 f"first chunk holding it (of {TOP_K})": f"{round(1 / result['MRR'])}" if result['MRR'] else "-",
                 f"recall@{TOP_K}": f"{result['recall']:.0%}",
                 "files holding it": len(holders[i])})
left_aligned(pd.DataFrame(rows).set_index("Q #"))


In [ ]:
#@title One miss, close up
# The first sampled question whose passage did not reach the top 5. No model call: the question
# was embedded once above, and the bundle carries its vector.
missed = [i for i in sample.index if not at_k.loc[i + 1, 'hit']]
if not missed:
    print(f"Every passage in this sample arrived in the top {TOP_K}. The misses are in the other "
          f"{int((~at_k['hit']).sum())} of {len(frame)} questions.")
else:
    i = missed[0]
    arrived = index.as_retriever(similarity_top_k=TOP_K).retrieve(bundles[i])
    print("Question:", frame.loc[i, 'question'])
    print("\nThe passage it was written from sits in:")
    print("   ", ", ".join(holders[i]) if holders[i] else "no chunk of this index (it was cut across chunks, or lost in extraction)")
    print(f"\nThe {TOP_K} chunks that arrived came from:")
    for r in arrived:
        print(f"    {r.node.metadata['file_name']}, page {r.node.metadata.get('page_label', '?')}")
    if set(holders[i]) & {r.node.metadata['file_name'] for r in arrived}:
        print("\nA chunk from a file that holds the passage did arrive, but it was not the chunk that holds it.")
    print("\nThe passage:", " ".join(frame.loc[i, 'passage'].split())[:400])


With every golden question scored, we can say how well this system retrieves. There are many more ways to evaluate it and to improve it, and Lab 3 tries several.

**What these numbers are for.** They are a baseline for this index, not a target to beat in the next lab. Measured on every golden question that scores 1.0, with the top 5 chunks (2 runs, 2026-10-21):

```
hit@5  61.4%    source hit@5  59.6%    recall@5  60.3%    MRR  0.450
```

The same index at deeper cuts (2 runs, 2026-10-21), with the mean tokens of chunk text sent per question:

```
k = 10   hit 70.2%   source hit 70.2%   recall 69.2%   MRR 0.463   2,902 tokens
k = 20   hit 84.2%   source hit 84.2%   recall 81.0%   MRR 0.472   5,663 tokens
```

Three of the 57 passages are not held by any chunk of this index (54 of 57 are), so no retriever could return them.

**Expect your numbers to match, give or take a question.** Retrieval over fixed vectors repeats itself: both of our runs gave identical numbers. But the questions are embedded fresh every run, and re-embedding can move a near tie, so yours may differ from this baseline by one question, which is 1.8 points on 57 questions. A gap bigger than that is worth a look.

You should see roughly **326 chunks** indexed from 186 pages (2 runs, 2026-10-21). Lab 3 downloads a corpus of 255 chunks over the same documents. Both are correct: this notebook chunks the PDFs itself, and Lab 3 uses a corpus parsed separately with LlamaParse. Different parse, different chunk boundaries, same source material.

### Where this system fails, and how it differs from the paint one

The BEHR version of this notebook, on the same four numbers (2 runs, 2026-10-21):

```
hit@5  70.5%    source hit@5  38.0%    recall@5  72.6%    MRR  0.538
```

The benefits index is lower on hit, recall and MRR, but far higher on source hit: 59.6% against 38.0%. The reason is the documents. Only 3 of the 57 benefits passages appear in more than one file, against 134 of 166 on the paint data sheets, so when a benefits chunk arrives it is almost always from the right file. The paint system hands over the right words from the wrong sheet. This one more often hands over nothing that answers the question: about four questions in ten at k = 5. Asking for 20 chunks instead of 5 closes much of that gap (hit 84.2%) at nearly four times the tokens.

An average also hides the questions that matter most. Lab 3 is for finding *where* this system fails, and two traps are already waiting in this corpus: the retiree and active dental handbooks share about half their text, and one of the 20 documents is for 2027, not 2026.

**Lab 3 runs this same scoring cell on its own corpus.** It scores naive retrieval against hybrid retrieval on the corpus parsed with LlamaParse, on these same 57 questions, and prints this baseline beside its own, so there is nothing for you to write down or carry over.

<!-- twin: dataset-specific -->

____
## LAB 2-C: Make your own test questions (about 30 minutes)

The golden set in 2-B was generated by a tool, and you only read a few of its questions. A test set you can trust is one a person has read. Here you build a small one from the hard pages you just inspected, review it, and use it to compare two retrievers.

1. **Generate** (about 10 minutes): ask the model for five questions per page, each with the passage that answers it.
2. **Review** (about 10 minutes): keep, fix or drop each question, and say why.
3. **Experiment** (about 10 minutes): score naive and hybrid retrieval on the questions you kept. No judge model is involved.
4. **Hold out**: tune on one half of the documents and report once on the other.

A useful test question records the evidence it needs: here, the quote that answers it and the page it sits on. A question that a program has checked is a development label, not gold.

### Step 1. Generate

By default the pages are the highest-scoring ones from your triage, one page from each of four different documents (any page you gave a verdict comes first). A page with almost no indexed text is skipped and the cell says so. To use any page of any document instead, replace `GEN_PAGES` in the cell with a list of `(file name, page number)` pairs; the example is in a comment.

There is one model call per page, with the same model as the rest of this lab. It is given the text this lab indexes for that page, the pypdf text, so a question inherits whatever pypdf got wrong. The prompt is short enough to read in the cell.

Then a program checks each quote: does at least 80% of it appear on the page? Questions whose quote does not are flagged **quote not found on the page**. That is a hint for your review, not a verdict. An automatic check like this gives you development labels, not gold: a quote can be on the page and still not answer the question.

In [ ]:
#@title Generate five questions per page (about 10 minutes)
import re

# Does this quote really appear in the text? The same rule scores retrieval further down: it is
# the shared cell from 2-B (share_found), which counts a quote as found when enough of its
# 6-word runs appear in the text.

# Which pages? By default: one page from each of four different documents, taken from the
# pages you gave a verdict, then from the triage list. A page with almost no indexed text is
# skipped, because a question needs a passage to quote. To use your own pages, replace
# GEN_PAGES with any list of (file name, page number), for example
#   GEN_PAGES = [("2026-coben-handbook.pdf", 7), ("2026-dental-active-handbook.pdf", 26)]
def default_pages(n_docs=4, min_chars=200):
    chosen = {}
    for doc, page_no in list(zip(page_verdicts["doc"], page_verdicts["page"])) + list(zip(flagged["doc"], flagged["page"])):
        if doc in chosen:
            continue
        if len(indexed_page_text(doc, int(page_no)).strip()) < min_chars:
            print(f"Skipping {doc} page {page_no}: this lab indexes almost no text for it, so a question has nothing to quote.")
            continue
        chosen[doc] = int(page_no)
        if len(chosen) == n_docs:
            break
    return list(chosen.items())

GEN_PAGES = default_pages()

# The whole prompt, short enough to read and change. The quote has to be copied word for word,
# because that quote is what we check the retrieved chunks against later.
GENERATE_PROMPT = """Below is the text of one page from a document.
Write 5 questions that a real person could type into a help desk search box and that this page answers.
For each question give the passage on the page that answers it, copied word for word.
Return a JSON object with one key, "questions": a list of 5 objects, each with the keys
"question", "quote" and "page" (the page number given below).

Document: {doc}
Page: {page}
Text of the page:
{text}"""

def generate_questions(doc, page_no):
    prompt = GENERATE_PROMPT.format(doc=doc, page=page_no, text=indexed_page_text(doc, page_no))
    reply = completion(model=MODEL, temperature=0, response_format={"type": "json_object"},
                       messages=[{"role": "user", "content": prompt}])
    try:  # models sometimes wrap JSON in a code fence, or return a bare list
        raw = re.sub(r"^```(?:json)?|```$", "", reply.choices[0].message.content.strip(), flags=re.M)
        data = json.loads(raw)
        items = data["questions"] if isinstance(data, dict) else data
        return [dict(doc=doc, page=page_no, question=str(q["question"]).strip(), quote=str(q["quote"]).strip())
                for q in items if isinstance(q, dict) and q.get("question") and q.get("quote")]
    except Exception:
        print(f"The reply for {doc} page {page_no} was not JSON we could read. Run this cell again.")
        return []

with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
    batches = list(pool.map(lambda p: generate_questions(*p), GEN_PAGES))
generated = pd.DataFrame([row for batch in batches for row in batch], columns=["doc", "page", "question", "quote"])

# The automatic check: is the quote on the page the model was given?
generated["quote_check"] = [
    "ok" if share_found(r.quote, [indexed_page_text(r.doc, r.page)]) >= 0.8 else "quote not found on the page"
    for r in generated.itertuples()]

print(f"{len(generated)} questions from {len(GEN_PAGES)} pages. "
      f"{(generated['quote_check'] != 'ok').sum()} have a quote that was not found on the page.")
pd.set_option("display.max_colwidth", 90)
generated[["doc", "page", "question", "quote_check"]]

### Step 2. Review

Read every question and its quote. Would a person type this? Does the quote really answer it? Record your decisions in the two dicts in the next cell: `VERDICTS` marks a row keep, fix or drop with a reason, and `FIXES` gives the rewritten text of a question you want to fix. Both start empty, which keeps every question as generated, so the cell runs as it is. The counts print beside the golden set's style counts from 2-B, so you can compare your questions with the generator's.

In [ ]:
#@title Review: keep, fix or drop each question
# Read the table above and the quotes below, then record your decisions in these two dicts.
# Both start empty, so everything is kept as generated. The number is the row number on the left.
#   VERDICTS = {3: ("drop", "answer is not on the page")}      # "keep", "fix" or "drop", and why
#   FIXES    = {7: "How many days do I have to submit a claim?"}   # your rewritten question
VERDICTS = {}
FIXES = {}

reviewed = generated.assign(original_question=generated["question"], verdict="keep", reason="")
for row_number, (verdict, reason) in VERDICTS.items():
    if row_number in reviewed.index and verdict in ("keep", "fix", "drop"):
        reviewed.loc[row_number, ["verdict", "reason"]] = [verdict, reason]
    else:
        print(f"Ignored VERDICTS entry {row_number}: use a row number from the table and keep, fix or drop.")
for row_number, new_question in FIXES.items():
    if row_number in reviewed.index:
        reviewed.loc[row_number, "question"] = new_question
        if reviewed.loc[row_number, "verdict"] != "drop":
            reviewed.loc[row_number, "verdict"] = "fix"
    else:
        print(f"Ignored FIXES entry {row_number}: there is no such row.")

counts = reviewed["verdict"].value_counts().reindex(["keep", "fix", "drop"], fill_value=0)
print("Your review of the questions you generated:")
print(counts.to_string())
print("\nFor comparison, the styles of the golden set's questions (from the cell above):")
print(style_counts.to_string())
print("\nThe generator wrote those styles on request. Which of yours would a real user type?")

kept_questions = reviewed[reviewed["verdict"] != "drop"]
pd.set_option("display.max_colwidth", 200)
reviewed[["page", "question", "quote", "quote_check", "verdict", "reason"]]

### Step 3. Experiment

Two retrievers, the same chunks you indexed in 2-A:

* **naive**: dense retrieval, the retriever from 2-A. It ranks chunks by embedding similarity alone.
* **hybrid**: the dense ranking and a BM25 keyword ranking fused by reciprocal rank, built exactly as Lab 3 builds it. Lab 3 explains the fusion.

Each is scored on your kept questions with the four numbers from 2-B, computed by the same shared cell: hit@5, source hit@5, recall@5 and MRR. The quote you recorded is the passage, and the page's document is the source file. A chunk holds the quote when at least `hit_threshold` of the quote's `shingle_size`-word runs appear in it, after lowercasing and stripping markup and punctuation. A quote shorter than `shingle_size` words counts when its text appears inside the chunk. Both settings come from the shared config, the same ones Lab 3 reads.

With a handful of questions the gap between two retrievers is often one or two questions. Read the table as a first look.


In [ ]:
#@title Experiment: naive against hybrid retrieval on your kept questions
import logging, nest_asyncio, Stemmer
from llama_index.core.llms import MockLLM
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever

logging.getLogger("bm25s").setLevel(logging.WARNING)   # the BM25 library logs every index it builds
nest_asyncio.apply()  # lets the fusion retriever run its two searches together inside a notebook

# naive: the dense retriever from 2-A. keyword: BM25 over the same chunks you indexed.
# hybrid: fuse the two ranked lists by reciprocal rank, built as Lab 3 builds it.
# num_queries=1 means no model is asked to rewrite the question. The retriever still wants an
# llm object, so it gets one that cannot be called.
def engines_at(k):
    dense = index.as_retriever(similarity_top_k=k)
    keyword = BM25Retriever.from_defaults(nodes=index_nodes, similarity_top_k=k,
                                          stemmer=Stemmer.Stemmer("english"), language="english")
    hybrid = QueryFusionRetriever([dense, keyword], mode="reciprocal_rerank", num_queries=1,
                                  similarity_top_k=k, use_async=True, llm=MockLLM())
    return {"naive (dense)": dense, "hybrid (dense + BM25)": hybrid}

# Your questions in the shape the shared scoring cell expects: the quote is the passage, and the
# page's document is the source file.
mine = kept_questions.rename(columns={"quote": "passage", "doc": "source"}).reset_index(drop=True)
my_bundles = embed_questions(mine["question"])

# The only model calls are query embeddings. depths=[TOP_K] scores the top 5 only.
per_question = score_engines(engines_at, mine, my_bundles, text_of=lambda r: r.node.get_content(), depths=[TOP_K])
per_question["doc"] = per_question["Q #"].map(lambda n: mine.loc[n - 1, "source"])   # for the hold-out split

print(f"Scored {len(mine)} kept questions on {per_question['engine'].nunique()} retrievers (top {TOP_K} chunks each).")
summary_at(per_question, TOP_K)


### Step 4. Hold out

The next cell splits your kept questions **by document**, not by question. Questions written from the same page are near copies of each other, and a split by question would let the half you tune on leak into the half you report on. It picks the retriever that wins on the tune half, then shows that one retriever on the held-out half.

Look at the held-out half once. Every time you peek at it and change something, it becomes a second tuning set, and the number you report no longer says how the choice will do on questions you have not seen.

In [ ]:
#@title Hold out: tune on one half of the documents, report once on the other
# Split by document, not by question. Questions written from the same page are near copies of
# each other, and a split by question would let the tuning half leak into the held-out half.
documents_in_set = sorted(mine["source"].unique())
tune_docs, held_docs = documents_in_set[0::2], documents_in_set[1::2]
print(f"Tune half: {len(tune_docs)} document(s). Held-out half: {len(held_docs)} document(s).")

if not held_docs:
    print("Your kept questions come from a single document, so there is nothing to hold out. "
          "Generate from pages of at least two documents and run the cells again.")
else:
    tune = summary_at(per_question[per_question["doc"].isin(tune_docs)], TOP_K)   # naive is scored first
    print("\nTune half, both engines:")
    display(tune)
    # Pick on MRR, then hit@k. A tie goes to the simpler engine, naive.
    winner = tune.sort_values([f"MRR@{TOP_K}", f"hit@{TOP_K}"], ascending=False, kind="stable").index[0]
    print(f"Winner on the tune half: {winner}. Now the held-out half, once:")
    display(summary_at(per_question[per_question["doc"].isin(held_docs) & (per_question["engine"] == winner)], TOP_K))


---
**STOP** - end of LAB 2-C

### What you should have seen

* **How many chunks the index holds.** The data ingestion cell printed 326 chunks from 186 pages (2 runs, 2026-10-21). The exact count depends on the chunk size, the text pypdf extracts and which pages have any text, so a different setup can move it by a few.

* **An answer can look right and cite the wrong document.** The retiree and active dental handbooks share about half their text word for word, and one of the 20 documents is for 2027, not 2026. Look at the sources printed under your answers: right topic, wrong handbook or wrong year is the most expensive kind of wrong, because the answer still reads well.

* **The hardest pages in the corpus, and why each was flagged.** The triage scored every page with PyMuPDF only. Its ten highest-scoring pages (1 run, 2026-10-20), at most four per document and three alike:
    * `2026-long-term-disability-benefits-guide.pdf` page 5, score 9: tables found; merged header cells; wrapped cell (4+ lines); ligature glyphs / dropped letters
    * `2026-long-term-disability-benefits-guide.pdf` page 12, score 9: tables found; wrapped cell (4+ lines); number-dense lines
    * `2026-long-term-disability-benefits-guide.pdf` page 7, score 8: tables found; wrapped cell (4+ lines); number-dense lines; ligature glyphs / dropped letters
    * `2026-long-term-disability-benefits-guide.pdf` page 10, score 8: tables found; merged header cells; number-dense lines; ligature glyphs / dropped letters
    * `2026-dental-active-handbook.pdf` page 26, score 7: tables found; merged header cells; wrapped cell (4+ lines); number-dense lines
    * `2026-dental-retiree-handbook.pdf` page 16, score 7: tables found; merged header cells; number-dense lines
    * `2026-dental-retiree-handbook.pdf` page 26, score 7: tables found; merged header cells; wrapped cell (4+ lines); number-dense lines
    * `2027-benefit-summary-guide-active-state-employees.pdf` page 16, score 7: tables found; merged header cells; wrapped cell (4+ lines); number-dense lines
    * `2027-benefit-summary-guide-active-state-employees.pdf` page 23, score 7: tables found; wrapped cell (4+ lines); number-dense lines
    * `2026-coben-handbook.pdf` page 7, score 6: tables found; number-dense lines

* **What the page images showed.** Four of the ten, looked at beside their text (1 run, 2026-10-20). On the long term disability guide (page 5) the page reads cleanly, but pypdf writes glyph names such as `/f_i` where the "fi" ligature was ("Bene/f_it", "/f_irst"). The ingestion cell now puts those letters back before indexing, so a search for "benefit" can match those words. The two coverage option headings still run into the text beside them. On the CoBen premium table (page 7), pypdf keeps each plan with its three prices on one line, while PyMuPDF puts every price on a line of its own, so only line order ties a price to a plan: a page that is hard for one reader and fine for the other. On the retiree dental handbook (page 26) and the 2027 summary guide (page 16) the wide plan tables survive well in the indexed text, one row per line in order, so the check by eye is the multi-line cells ("Retirees and Dependents", the orthodontia wording) and whether each percentage still sits under the right plan.

* **What the retrieval baseline is.** Over every golden question that scores 1.0 (57), the top 5 chunks hold the passage for 61.4% of them (hit@5), carry 60.3% of the passage between them (recall@5) and put the first holding chunk at a mean reciprocal rank of 0.450 (2 runs, 2026-10-21). Asking for 10 chunks adds 8.8 points of hit and 8.9 of recall for about twice the tokens. Source hit@5 is 59.6%, close to hit@5, because almost every passage sits in one file only.

* **What taking the metadata out of the embedded text was worth.** By default llama-index writes `page_label` and `file_path` into every chunk it embeds. Keeping them out raised dense hit@5 from 56.1% to 61.4% and MRR from 0.403 to 0.450 on this corpus (1 run, 2026-10-21). With 57 questions the hit@5 gain is inside the noise: its 95% interval runs from -3.5 to +14.0 points.

* **Your own test set.** Lab 2-C ends with your kept, fixed and dropped counts, and one retriever reported once on a held-out half. Your numbers are your own: a handful of questions, split by document, so compare the shape of the result with your neighbour's rather than the third decimal.


<!-- twin: dataset-specific -->